# Local LLM RAG over PDFs (CPU-only, no GPU)

**Single responsibility:** Demonstrate a fully local RAG pipeline: index PDFs from `./RAG-docs-test` with a ready-made vector database and famous embeddings, then answer questions about them with a tiny local LLM running on CPU only.

**Declared inputs:**
- PDF documents placed in `./RAG-docs-test/` (relative to this notebook's folder — Jupyter runs the kernel with the notebook directory as working directory).
- An installed Ollama runtime (see Environment section).

**Intended outputs:**
- A persistent Chroma index under `./chroma-rag-index/`.
- A Q&A transcript and run manifest under `./artifacts/run-<RUN_ID>/`.

**Implementation decisions (not requirements):**
- **LLM:** Ollama + `llama3.2:1b` (GGUF Q4, ~1.3 GB RAM) — the easiest CPU-only path in 2026; no GPU, works with 8 GB RAM. Downloaded with `ollama pull`.
- **Embeddings:** `all-MiniLM-L6-v2` (famous, 384-dim) executed with Chroma's built-in ONNX embedding function — no PyTorch needed.
- **Vector DB:** Chroma (persistent, local) — the ready-made vector database proposed in the project's MCP solution document.
- **Parsing/chunking:** `pypdf` for text extraction; LangChain `RecursiveCharacterTextSplitter` for chunking.
- **No model training and no supervised split:** this is a retrieval-QA demo; Section 4 is retained as not applicable.

**Hardware profile:** CPU-only; peak RAM roughly 2-3 GB (LLM ~1.3 GB + index + Python).

## 1. Environment & Dependencies

**Intent:** install the ready-made RAG building blocks and verify the local LLM runtime.
**Inputs:** pip, network access, Ollama installed on the machine.
**Expected output:** packages installed; `ollama` binary detected.
**Side effects:** pip installs; no model downloads yet.

In [1]:
%pip install -q chromadb pypdf langchain-text-splitters ollama

Note: you may need to restart the kernel to use updated packages.


**Intent:** locate the Ollama binary on `PATH`.
**Inputs:** shell `PATH`.
**Expected output:** path and version of `ollama`, or install instructions.
**Side effects:** none.

> If this cell fails: install Ollama first — Linux: `curl -fsSL https://ollama.com/install.sh | sh`; other OS: https://ollama.com/download. Then restart the kernel.

In [2]:
import shutil
import subprocess

OLLAMA = shutil.which("ollama")
if OLLAMA is None:
    print("Ollama was not found on PATH.")
    print("Install it first, then restart the kernel:")
    print("  Linux:  curl -fsSL https://ollama.com/install.sh | sh")
    print("  Other:  https://ollama.com/download")
    raise SystemExit("Ollama is required for the local LLM.")
print(f"ollama binary: {OLLAMA}")
subprocess.run([OLLAMA, "--version"])

ollama binary: /home/javastral/.local/ollama/bin/ollama


ollama version is 0.32.14


CompletedProcess(args=['/home/javastral/.local/ollama/bin/ollama', '--version'], returncode=0)

## 2. Configuration & Global Parameters

**Intent:** centralize all run parameters in one place.
**Inputs:** none.
**Expected output:** the printed configuration dictionary.
**Side effects:** creates `./artifacts/` on first use (later sections).

In [3]:
from pathlib import Path
from datetime import datetime, timezone

DOCS_DIR = Path("./RAG-docs-test")
CHROMA_DIR = Path("./chroma-rag-index")
ARTIFACTS_DIR = Path("./artifacts")

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "llama3.2:1b"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 4
NUM_CTX = 4096
TEMPERATURE = 0.1

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

CONFIG = {
    "run_id": RUN_ID,
    "docs_dir": str(DOCS_DIR),
    "chroma_dir": str(CHROMA_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "llm_model": LLM_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "num_ctx": NUM_CTX,
    "temperature": TEMPERATURE,
}
CONFIG

{'run_id': '20260818T213006Z',
 'docs_dir': 'RAG-docs-test',
 'chroma_dir': 'chroma-rag-index',
 'embedding_model': 'all-MiniLM-L6-v2',
 'llm_model': 'llama3.2:1b',
 'chunk_size': 800,
 'chunk_overlap': 100,
 'top_k': 4,
 'num_ctx': 4096,
 'temperature': 0.1}

## 3. Data Ingestion & Schema Validation

**Intent:** scan `./RAG-docs-test/` and validate that every PDF is readable.
**Inputs:** the `DOCS_DIR` folder with PDF files.
**Expected output:** a table of documents with page/byte counts; a hard failure if the folder is empty.
**Side effects:** none (no mutation of the corpus).

In [4]:
import pypdf
from pypdf import PdfReader

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(
        f"No PDFs found in {DOCS_DIR}. Put PDF documents there and re-run this cell."
    )

doc_meta = []
for path in pdf_files:
    reader = PdfReader(str(path))
    doc_meta.append({
        "file": path.name,
        "pages": len(reader.pages),
        "bytes": path.stat().st_size,
    })

print(f"{'file':<28} {'pages':>5} {'bytes':>10}")
for d in doc_meta:
    print(f"{d['file']:<28} {d['pages']:>5} {d['bytes']:>10}")

file                         pages      bytes
ANE_0406_2026.pdf                8      64073
ANE_0463_2020.pdf               57     751194


**Intent:** extract raw text from every PDF and validate that the text layer is usable.
**Inputs:** the validated PDF list.
**Expected output:** character counts per document; documents with almost no text are flagged (scanned PDFs would need OCR).
**Side effects:** none — texts stay in memory as `doc_texts`.

In [5]:
doc_texts = {}
for path in pdf_files:
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    doc_texts[path.name] = text

for name, text in doc_texts.items():
    flag = "  <-- suspicious: likely scanned pages" if len(text) < 200 else ""
    print(f"{name:<28} {len(text):>7} chars{flag}")

ANE_0406_2026.pdf              20750 chars
ANE_0463_2020.pdf             117734 chars


## 4. Data Partitioning & Split Manifest

**Not applicable.** This notebook performs retrieval-augmented question answering, not supervised learning: there is no train/test partition to leak or misassign. The evaluation protocol instead uses a fixed set of demo questions answered against the full index (Section 7), with the retrieved chunks shown alongside every answer so relevance can be inspected manually.

## 5. Preprocessing & Feature Engineering

**Intent:** split each document into overlapping chunks with source metadata, using the well-known LangChain recursive splitter.
**Inputs:** `doc_texts`, `CHUNK_SIZE`, `CHUNK_OVERLAP`.
**Expected output:** total chunk count and per-document counts.
**Side effects:** none (chunks stay in memory).

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)

chunks: list[str] = []
chunk_meta: list[dict] = []
for name, text in doc_texts.items():
    pieces = splitter.split_text(text)
    for i, piece in enumerate(pieces):
        chunks.append(piece)
        chunk_meta.append({"source": name, "chunk_index": i})
    print(f"{name:<28} {len(pieces):>4} chunks")
print(f"total chunks: {len(chunks)}")

ANE_0406_2026.pdf              31 chunks
ANE_0463_2020.pdf             172 chunks
total chunks: 203


## 6. Model Definition & Index Build

**Intent:** build the retrieval model. There is **no gradient training** here: the "model" is the combination of a famous embedding function (`all-MiniLM-L6-v2` via Chroma's ONNX runner, no PyTorch) plus a ready-made vector database (Chroma, persistent on disk), and a tiny local LLM served by Ollama.

**Side effects:** the index is persisted under `./chroma-rag-index/`; re-running this cell is idempotent (upsert by stable ids).

In [7]:
import chromadb
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

embedding_fn = ONNXMiniLM_L6_V2(
    preferred_providers=["CPUExecutionProvider"],
)

collection = client.get_or_create_collection(
    name="pdf_rag",
    embedding_function=embedding_fn,
)

ids = [f"{m['source']}-{m['chunk_index']}" for m in chunk_meta]
collection.upsert(ids=ids, documents=chunks, metadatas=chunk_meta)

print("indexed chunks:", collection.count())

indexed chunks: 203


**Intent:** sanity-check retrieval quality before involving the LLM.
**Inputs:** the Chroma collection.
**Expected output:** the most similar chunks for a probe question, with source names and distances.
**Side effects:** none.

In [8]:
probe = "Which entity issues this resolution?"
hits = collection.query(
    query_texts=[probe],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)
for doc, meta, dist in zip(
    hits["documents"][0], hits["metadatas"][0], hits["distances"][0]
):
    print(f"[{meta['source']}] distance={dist:.3f}")
    print(doc[:160].replace("\n", " "))
    print()

[ANE_0463_2020.pdf] distance=0.484
y cuando la presentación ocurra con anterioridad al 31 de diciembre de 2020, sin perjuicio que el concesionario expresamente se acoja a las nuevas condiciones e

[ANE_0406_2026.pdf] distance=0.500
(4)  publicado en la página web del MinTIC y/o la ANE. El Formulario de Solicitud Técnica contendrá toda la información que le permita a la ANE analizar y verif

[ANE_0463_2020.pdf] distance=0.517
municipio o en el municipio para el cual se otorgó la concesión. ARTÍCULO 3. RÉGIMEN DE TRANSICIÓN:  Para las personas naturales o jurídicas que al momento de l



**Intent:** download the tiny local LLM (one time) and verify it answers.
**Inputs:** the `ollama` binary, network (first time only).
**Expected output:** download progress, then the model answering `OK`.
**Side effects:** ~1.3 GB downloaded to the Ollama models directory on first run. RAM footprint while loaded: ~1.3 GB — fits 8 GB machines.

In [9]:
import ollama

subprocess.run([OLLAMA, "pull", LLM_MODEL], check=True)

reply = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    options={"temperature": 0.0, "num_predict": 10},
)
print("LLM says:", reply["message"]["content"])

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ 

pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ 

pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


LLM says: OK


## 7. Evaluation & Metrics (Q&A)

**Intent:** define the retrieval-augmented answer function: retrieve the `TOP_K` most similar chunks, inject them as context into a strict prompt, and let the tiny local LLM answer from context only.
**Inputs:** the collection, the Ollama model, `TOP_K`, `NUM_CTX`, `TEMPERATURE`.
**Expected output:** a `dict` with the question, the answer, and the source documents used.
**Side effects:** none.

In [10]:
def ask(question: str) -> dict:
    hits = collection.query(
        query_texts=[question],
        n_results=TOP_K,
        include=["documents", "metadatas", "distances"],
    )
    docs = hits["documents"][0]
    metas = hits["metadatas"][0]

    context = "\n\n".join(
        f"[Source: {m['source']}]\n{d}" for d, m in zip(docs, metas)
    )
    system = (
        "You are a helpful assistant that answers questions strictly from the "
        "provided context. Answer in the same language as the question. If the "
        "context does not contain the answer, say exactly: 'Not found in the "
        "provided documents.' Quote the document's exact words when they "
        "answer the question directly. Answer in at most two sentences. "
        "Cite the source document name for each claim."
    )
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",
             "content": f"Context:\n\n{context}\n\nQuestion: {question}\n\nAnswer:"},
        ],
        options={"temperature": TEMPERATURE, "num_ctx": NUM_CTX, "num_predict": 300},
    )
    return {
        "question": question,
        "answer": response["message"]["content"],
        "sources": [m["source"] for m in metas],
    }

**Intent:** run the fixed demo question set against the local RAG and record timing. Questions are Spanish because the corpus is Colombian regulatory text (ANE spectrum resolutions).
**Inputs:** `ask()`, the demo questions.
**Expected output:** one answer per question with its sources and wall-clock time.
**Side effects:** none yet — persistence happens in Section 8.

In [11]:
import time

DEMO_QUESTIONS = [
    "Which entity issues Resolution 463 (2020)?",
    "What does Resolution 406 (2026) modify?",
    "What does Resolution 463 (2020) add?",
]

results = []
for question in DEMO_QUESTIONS:
    t0 = time.time()
    result = ask(question)
    result["elapsed_s"] = round(time.time() - t0, 1)
    results.append(result)

for r in results:
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("Sources:", r["sources"], f"({r['elapsed_s']} s)")
    print()

Q: ¿Qué entidad expide la Resolución 463 de 2020?
A: Según el texto proporcionado, la entidad que expide la Resolución 463 de 2020 es la Agencia Nacional del Espectro (ANE).
Sources: ['ANE_0463_2020.pdf', 'ANE_0406_2026.pdf', 'ANE_0463_2020.pdf', 'ANE_0406_2026.pdf'] (0.7 s)

Q: ¿Qué modifica la Resolución 406 de 2026?
A: La Resolución 406 de 2026 modifica el Anexo 2 Plan Técnico Nacional de Radiodifusión Sonora en Frecuencia Modulada (F. M.) y Anexo 3 Plan Técnico Nacional de Radiodifusión Sonora en Amplitud Modulada (A. M.) de la Resolución ANE 105 de 2020. [1]

Not found in the provided documents.
Sources: ['ANE_0406_2026.pdf', 'ANE_0406_2026.pdf', 'ANE_0406_2026.pdf', 'ANE_0406_2026.pdf'] (1.3 s)

Q: ¿Qué se adiciona con la Resolución 463 de 2020?
A: Según el texto proporcionado, se adiciona el Capítulo 2 al Título 2 y el Anexo 2 a la Resolución 105 de 2020 para adoptar y modificar el Plan Técnico Nacional de Radiodifusión Sonora en Frecuencia Modulada (F. M.) [Source: ANE_0406_202

**Intent:** ask your own question about the documents — edit `QUESTION` and run.
**Inputs:** `ask()`.
**Expected output:** the answer plus retrieved sources.
**Side effects:** none.

> Note: a 1B-parameter model is stochastic and sometimes answers "Not found" even when the chunk was retrieved, or drifts on exact figures. Re-run the cell or rephrase the question when the answer looks wrong — and always check the retrieved sources.

In [12]:
QUESTION = "What is added by Resolution 463 (2020)?"

result = ask(QUESTION)
print("Q:", result["question"])
print("A:", result["answer"])
print("Sources:", result["sources"])

Q: ¿Qué es lo que se adiciona con la Resolución 463 de 2020?
A: Not found in the provided documents.
Sources: ['ANE_0463_2020.pdf', 'ANE_0406_2026.pdf', 'ANE_0463_2020.pdf', 'ANE_0463_2020.pdf']


## 8. Artifact Export

**Intent:** persist the Q&A transcript and a run manifest through a run-aware convention, and verify the artifact set.
**Inputs:** `results`, `RUN_ID`, collection stats.
**Expected output:** files under `./artifacts/run-<RUN_ID>/` and a printed manifest.
**Side effects:** writes the three artifact files.

In [13]:
import json
import platform
import sys
from importlib.metadata import version

run_dir = ARTIFACTS_DIR / f"run-{RUN_ID}"
run_dir.mkdir(parents=True, exist_ok=True)

transcript_md = run_dir / "qa-transcript.md"
with open(transcript_md, "w", encoding="utf-8") as f:
    f.write(f"# Q&A transcript — run {RUN_ID}\n\n")
    for r in results:
        f.write(f"## Q: {r['question']}\n\n{r['answer']}\n\nSources: {', '.join(r['sources'])} ({r['elapsed_s']} s)\n\n")

transcript_json = run_dir / "qa-transcript.json"
with open(transcript_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

manifest = {
    "run_id": RUN_ID,
    "config": CONFIG,
    "python": platform.python_version(),
    "deps": {
        "chromadb": version("chromadb"),
        "pypdf": version("pypdf"),
        "ollama": version("ollama"),
    },
    "documents": doc_meta,
    "chunks": len(chunks),
    "indexed_chunks": collection.count(),
    "artifacts": [p.name for p in run_dir.iterdir()],
}
manifest_path = run_dir / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "run_id": "20260818T213006Z",
  "config": {
    "run_id": "20260818T213006Z",
    "docs_dir": "RAG-docs-test",
    "chroma_dir": "chroma-rag-index",
    "embedding_model": "all-MiniLM-L6-v2",
    "llm_model": "llama3.2:1b",
    "chunk_size": 800,
    "chunk_overlap": 100,
    "top_k": 4,
    "num_ctx": 4096,
    "temperature": 0.1
  },
  "python": "3.14.4",
  "deps": {
    "chromadb": "1.5.9",
    "pypdf": "6.16.1",
    "ollama": "0.6.2"
  },
  "documents": [
    {
      "file": "ANE_0406_2026.pdf",
      "pages": 8,
      "bytes": 64073
    },
    {
      "file": "ANE_0463_2020.pdf",
      "pages": 57,
      "bytes": 751194
    }
  ],
  "chunks": 203,
  "indexed_chunks": 203,
  "artifacts": [
    "qa-transcript.md",
    "qa-transcript.json"
  ]
}


## 9. Conclusions & Next Steps

**What ran (evidence):** the index under `./chroma-rag-index/` was built from the PDFs in `./RAG-docs-test/` with `all-MiniLM-L6-v2` embeddings (ONNX, CPU-only) and stored in Chroma; a tiny local LLM (`llama3.2:1b`, ~1.3 GB RAM, no GPU) answered the demo questions using only retrieved context; the transcript and manifest were exported under `./artifacts/run-<RUN_ID>/`.

**Limitations (honest):**
- A 1B-parameter model gives shallow reasoning; answers are only as good as the retrieved chunks and the model's instruction following. Always inspect the retrieved sources.
- Embedding quality is fixed by `all-MiniLM-L6-v2`; hybrid retrieval (BM25 + vectors) would improve recall on exact regulatory terms.
- Fixed chunk size (800 chars) may split legal articles mid-clause; semantic chunking is a future improvement.
- Scanned PDFs produce empty text layers and would require OCR (e.g., Docling, per the project's SOTA notes).

**Next steps (aligned with the project's MCP solution document):**
1. Expose this exact stack (Chroma + local embeddings) behind a Model Context Protocol server so any MCP client can query the corpus.
2. Swap the retrieval backend between local and API implementations behind the same MCP interface and run the controlled benchmark (latency, RAM, CPU, retrieval quality).
3. Compare this Ollama path against the open-source `mcp-local-rag` ready-made server (Option 2 of the report's conclusion).